In [ ]:
import sys
import numpy as np

sys.path.append("..")
from src.qubo_windfarm_layout.model import load_wake_loss_data
from src.qubo_windfarm_layout.penalties import suggest_cardinality_penalty_from_layout, layout_yaml_to_z, suggest_spacing_penalty_from_layout, compute_lambda_from_lb, sdp_fixed_cardinality, sdp_spacing_violation

In [11]:
GRID_RESOLUTION = 100
TIMEOUT = 3600
REFERENCE_LAYOUT = f"../results/layouts/cpsat_{GRID_RESOLUTION}_{TIMEOUT}s.yaml"

# 1. Problem Based Penalty Calibration

In [3]:
lambda_cardinality_pb = (
    suggest_cardinality_penalty_from_layout(
        REFERENCE_LAYOUT
    )
)

lambda_spacing_pb = (
    suggest_spacing_penalty_from_layout(
        REFERENCE_LAYOUT
    )
)

print(f"Lambda cardinality = {lambda_cardinality_pb}")
print(f"Lambda cardinality = {lambda_spacing_pb}")

Reference turbines: 81
Max marginal wake loss: 24,142.767
Safety factor: 1.5
Suggested lambda cardinality: 36,214.151
Reference turbines: 81
Reference pairwise wake objective: 882,505.366
Safety factor: 1.2
Suggested lambda spacing: 1,059,006.439
Lambda cardinality = 36214.15094613261
Lambda cardinality = 1059006.4391287481


# 2. Relaxation-based penalty calibration 

In [4]:
from src.qubo_windfarm_layout.model import get_farm_area, get_mask
from src.solvers.utils import get_invalid_pairs
from src.qubo_windfarm_layout.evaluation import load_layout_coordinates

MIN_DISTANCE = 396  # 2 * 198 m

_, farm_area = get_farm_area()
X_grid, Y_grid, mask = get_mask(farm_area=farm_area, grid_resolution=GRID_RESOLUTION)
candidate_locations = np.column_stack([X_grid[mask], Y_grid[mask]])

invalid_pairs = get_invalid_pairs(
    candidate_locations=candidate_locations,
    min_distance=MIN_DISTANCE,
)

print(f"Candidati totali : {len(candidate_locations)}")
print(f"Coppie non valide: {len(invalid_pairs)}")

Candidati totali : 905
Coppie non valide: 3152


In [5]:
# Step 1: compute upper bound using a feasible solution
z_reference, candidate_locations = layout_yaml_to_z(
    yaml_path=REFERENCE_LAYOUT,
    grid_resolution=GRID_RESOLUTION,
)

print(f"Candidati totali   : {len(candidate_locations)}")
print(f"Turbine selezionate: {z_reference.sum()}")

Candidati totali   : 905
Turbine selezionate: 81


In [ ]:
# Step 2: compute lower bound using a relaxed solution
wake_loss = load_wake_loss_data(f"../results/precomputed/wake_loss_{GRID_RESOLUTION}m.npz")
wake_loss_matrix = wake_loss["wake_loss_matrix"]

LB_cardinality = sdp_fixed_cardinality(
    wake_loss_matrix,
    n_turbines=80,
    max_iters=1000
)

LB_spacing = sdp_spacing_violation(
    wake_loss_matrix,
    invalid_pairs,
    n_turbines=81,
    max_iters=1000
)

(CVXPY) Sep 15 01:08:43 PM: Your problem has 819930 variables, 2461602 constraints, and 0 parameters.
(CVXPY) Sep 15 01:08:43 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Sep 15 01:08:43 PM: DCP verification time: 0.0003 seconds.
(CVXPY) Sep 15 01:08:43 PM: Expression tree has 10 nodes.
(CVXPY) Sep 15 01:08:43 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Sep 15 01:08:43 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Sep 15 01:08:43 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Sep 15 01:08:43 PM: Compiling problem (target solver=SCS).
(CVXPY) Sep 15 01:08:43 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ExactCone2Cone -> EliminateZeroSized -> ConeMatrixStuffing -> SCS
(CVXPY) Sep 15 01:08:43 PM: Applying reduction Dcp2Cone


(CVXPY) Sep 15 01:08:43 PM: Applying reduction CvxAttr2Constr
(CVXPY) Sep 15 01:08:43 PM: Applying reduction ExactCone2Cone


                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Sep 15 01:08:43 PM: Applying reduction EliminateZeroSized
(CVXPY) Sep 15 01:08:43 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Sep 15 01:08:44 PM: Applying reduction SCS
(CVXPY) Sep 15 01:08:44 PM: Finished problem compilation (took 1.710e+00 seconds).
(CVXPY) Sep 15 01:08:44 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.3.1 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 410870, constraints m: 2051637
cones: 	  z: primal zero / dual free vars: 906
	  l: linear vars: 1639860
	  s: psd vars: 410871, ssize: 1
settings: eps_abs: 1.0e-04, eps_rel: 1.0e-04, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 1000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 5
lin-sys:  sparse-indirect-scs
	  nnz(A): 2053445, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua

...

  1000| 1.13e-02  8.39e-01  2.38e-01  8.51e+05  1.71e-02  3.97e+02 
------------------------------------------------------------------
status:  solved (inaccurate - reached max_iters)
timings: total: 3.97e+02s = setup: 7.63e-01s + solve: 3.96e+02s
	 lin-sys: 1.70e+02s, cones: 1.98e+02s, accel: 1.03e+01s
------------------------------------------------------------------
objective = 850934.658388 (inaccurate)
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilat

(CVXPY) Sep 15 01:15:29 PM: Applying reduction CvxAttr2Constr
/tmp/ipykernel_36726/3637958168.py:102: UserWarning: Constraint #3 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  problem.solve(
(CVXPY) Sep 15 01:15:29 PM: Applying reduction ExactCone2Cone
(CVXPY) Sep 15 01:15:29 PM: Applying reduction EliminateZeroSized
(CVXPY) Sep 15 01:15:29 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Sep 15 01:20:07 PM: Applying reduction SCS
(CVXPY) Sep 15 01:20:07 PM: Finished problem compilation (took 2.779e+02 seconds).
(CVXPY) Sep 15 01:20:07 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.3.1 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 410870, constraints m: 2051638
cones: 	  z: primal zero / dual free vars: 906
	  l: linear vars: 1639861
	  s: psd vars: 410871, ssize: 1
settings: eps_abs: 1.0e-04, eps_rel: 1.0e-04, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 1000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 5
lin-sys:  sparse-indirect-scs
	  nnz(A): 2056597, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua

/tmp/ipykernel_36726/3637958168.py:102: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  problem.solve(
(CVXPY) Sep 15 01:30:28 PM: Problem status: optimal_inaccurate
(CVXPY) Sep 15 01:30:28 PM: Optimal value: 8.744e+05
(CVXPY) Sep 15 01:30:28 PM: Compilation took 2.779e+02 seconds
(CVXPY) Sep 15 01:30:28 PM: Solver (including time spent in interface) took 6.215e+02 seconds


  1000| 2.26e-02  5.31e+00  1.02e+00  8.74e+05  1.63e-02  6.10e+02 
------------------------------------------------------------------
status:  solved (inaccurate - reached max_iters)
timings: total: 6.10e+02s = setup: 8.17e-01s + solve: 6.09e+02s
	 lin-sys: 3.70e+02s, cones: 2.00e+02s, accel: 1.04e+01s
------------------------------------------------------------------
objective = 874373.156179 (inaccurate)
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


In [8]:
# Step 3: compute lambda
z_reference = layout_yaml_to_z(yaml_path=REFERENCE_LAYOUT, grid_resolution=GRID_RESOLUTION)[0]

lambda_cardinality = compute_lambda_from_lb(wake_loss_matrix=wake_loss_matrix, z_reference=z_reference, lower_bound=LB_cardinality)
lambda_spacing = compute_lambda_from_lb(wake_loss_matrix=wake_loss_matrix, z_reference=z_reference, lower_bound=LB_spacing)

In [10]:
print(f"Lambda cardinality = {lambda_cardinality}")
print(f"Lambda cardinality = {lambda_spacing}")

Lambda cardinality = 31570.588592252454
Lambda cardinality = 8131.698590035776
